In [1]:
from common.paths import get_avito_data_dpath
from common.logger import AVITO_SHOULD_SPLIT_LOGGER as logger

import pandas as pd
import numpy as np

In [2]:
data_fpath = get_avito_data_dpath() / "rnc_dataset_markup.json"
id_cat_map_fpath = get_avito_data_dpath() / "rnc_mic_key_phrases.csv"

data_df = pd.read_json(data_fpath)
id_cat_map_df = pd.read_csv(id_cat_map_fpath)

id_cat_map = dict(zip(id_cat_map_df["mcId"], id_cat_map_df["mcTitle"]))
print(id_cat_map)
data_df.head()

{101: 'Ремонт квартир и домов под ключ', 102: 'Сантехника', 103: 'Электрика', 104: 'Натяжные потолки', 105: 'Укладка плитки', 106: 'Поклейка обоев', 107: 'Малярные работы', 108: 'Штукатурные работы', 109: 'Напольные покрытия', 110: 'Гипсокартон', 111: 'Демонтажные работы'}


,itemId,sourceMcId,sourceMcTitle,description,targetDetectedMcIds,targetSplitMcIds,shouldSplit,caseType,split
0,1000001,101,Ремонт квартир и домов под ключ,"Всё виды строительных работ\r\nКачественно, в ...",[],[],False,no_other_microcategories_detected,NaN
1,1000002,101,Ремонт квартир и домов под ключ,Профессионально и качественно сделаем ремонт к...,"[102, 103, 105, 108, 109, 110]",[],False,other_microcategories_detected_but_not_split,NaN
2,1000003,101,Ремонт квартир и домов под ключ,"ремонт квартир, ванной комнате , балкон",[102],[],False,other_microcategories_detected_but_not_split,NaN
3,1000004,101,Ремонт квартир и домов под ключ,ЗBОНИТЕ KОHСУЛЬТАЦИЯ БЕCПЛАTНAЯ ПO ТУЛЬСKOЙ ОБ...,"[102, 103, 104, 105, 106, 107, 108, 109, 110]","[102, 103, 104, 105, 106, 107, 108, 109, 110]",True,split,NaN
4,1000005,101,Ремонт квартир и домов под ключ,Ремонт квартир любой сложности. Квартиры под к...,[],[],False,no_other_microcategories_detected,NaN


In [3]:
from avito.features import extract_should_split_features

X = extract_should_split_features(data_df)
y = data_df["shouldSplit"].values
assert len(X) == len(y)
logger.info(f"Размер X: {X.shape}, размер y: {y.shape}")
X.head()

2026-04-07 23:26:41,061 - avito-should-split - INFO - [SHOULD_SPLIT] Размер X: (2480, 21), размер y: (2480,)


,description_word_count,description_char_count,split_marker_count,complex_marker_count,marker_ratio,has_bullets,sentence_count,avg_word_len,punctuation_ratio,max_keyphrase_rapidfuzz_mc_101,...,max_keyphrase_rapidfuzz_mc_103,max_keyphrase_rapidfuzz_mc_104,max_keyphrase_rapidfuzz_mc_105,max_keyphrase_rapidfuzz_mc_106,max_keyphrase_rapidfuzz_mc_107,max_keyphrase_rapidfuzz_mc_108,max_keyphrase_rapidfuzz_mc_109,max_keyphrase_rapidfuzz_mc_110,max_keyphrase_rapidfuzz_mc_111,split_marker_near_keyphrase
0,13.0,85.0,0.0,0.0,0.0,0,2.0,5.384615,0.023529,52.380951,...,34.482758,36.799999,30.357143,33.333332,32.758621,33.928570,28.318584,33.043480,33.027523,0
1,107.0,803.0,0.0,3.0,0.0,0,18.0,6.222222,0.029888,100.000000,...,75.000000,66.666664,81.250000,76.190475,66.666664,62.068966,100.000000,57.142857,68.965515,0
2,6.0,39.0,0.0,0.0,0.0,0,1.0,6.400000,0.051282,74.285713,...,54.545456,58.333332,57.142857,66.666664,48.387096,52.173912,70.588234,50.000000,55.882355,0
3,169.0,1247.0,0.0,0.0,0.0,0,10.0,6.193750,0.034483,6.451613,...,5.008945,6.366048,5.008945,72.727272,4.817128,4.834378,4.472272,60.869564,4.834378,0
4,26.0,181.0,0.0,1.0,0.0,0,6.0,5.692307,0.044199,69.565216,...,61.538460,48.484848,48.484848,76.190475,66.666664,27.272728,48.484848,57.142857,48.484848,0


In [13]:
from avito.embeddings import SentenceTransformerEncoder, EncoderConfig

encoder = SentenceTransformerEncoder(EncoderConfig.from_default_yaml())

def build_text_to_encode(row):
    return f"""{row['description']}"""

texts_to_encode = data_df.apply(build_text_to_encode, axis=1).tolist()
text_embeddings = encoder.encode(texts_to_encode)
X = np.hstack([X.values, text_embeddings])

2026-04-07 23:18:54,167 - avito-embeddings - INFO - [AVITO/EMBEDDINGS] Пробую загрузить локальную модель: C:\Users\User\Desktop\dirs\Dev\hack-mfti\avito\checkpoints\rubert-mini-frida
Default prompt name is set to 'Classification'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.
2026-04-07 23:18:54,308 - avito-embeddings - INFO - [AVITO/EMBEDDINGS] Запуск encode для 2480 текстов


Batches:   0%|          | 0/78 [00:00<?, ?it/s]

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [26]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train, sample_weight=1000000)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [27]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(X_test)
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
print(report)
cm

              precision    recall  f1-score   support

       False       0.82      0.99      0.89       403
        True       0.44      0.04      0.08        93

    accuracy                           0.81       496
   macro avg       0.63      0.52      0.49       496
weighted avg       0.75      0.81      0.74       496



array([[398,   5],
       [ 89,   4]])

In [21]:
def build_few_shots():
    random_ids = data_df.sample(5, random_state=42).index.tolist()
    few_shots = []
    for idx in random_ids:
        row = data_df.loc[idx]
        few_shots.append(f"""Описание: {row['description']}
Ответ: {row['shouldSplit']}""")
    return "\n".join(few_shots)

instruction_prompt = f"""
Ты — эксперт по классификации объявлений об услугах на платформе Авито.

Тебе даётся описание объявления, опубликованного в категории «Ремонт квартир и домов под ключ».
Твоя задача — определить, упоминает ли селлер какие-либо услуги, которые он готов выполнять ОТДЕЛЬНО, вне комплексного ремонта.

КРИТЕРИИ для ответа True (нужно разделить):
- Селлер явно предлагает отдельные услуги: «также делаем электрику», «сантехнику выполняем отдельно»
- Есть перечисление самостоятельных работ с ценами или условиями для каждой
- Используются фразы: «отдельно», «также выполняем», «помимо основного», «принимаем заказы на»

КРИТЕРИИ для ответа False (не нужно разделять):
- Услуги упомянуты только как составляющие комплексного ремонта: «ремонт под ключ, включая электрику и сантехнику»
- Перечисление видов работ без явного предложения их по отдельности
- Краткое общее описание без конкретики по отдельным услугам

ФОРМАТ ОТВЕТА:
Строго одна строка: True или False
Пример: True
Пример: False

ПРИМЕРЫ:
{build_few_shots()}
"""
print(instruction_prompt)

from common.mistral import call_mistral, MistralCallConfig

def build_messages(row):
    return [
        {"role": "system", "content": instruction_prompt},
        {"role": "user", "content": f"Описание: {row['description']}"},
    ]

cfg = MistralCallConfig(
    models_list=["ministral-14b-latest"]
)

from tqdm.auto import tqdm

mistral_predictions = []
subsample_df = data_df.sample(20, random_state=43)

for _, row in tqdm(subsample_df.iterrows(), total=len(subsample_df)):
    messages = build_messages(row)
    response = call_mistral(cfg, messages=messages, temperature=0.0)
    mistral_predictions.append(response)

mistral_predictions


Ты — эксперт по классификации объявлений об услугах на платформе Авито.

Тебе даётся описание объявления, опубликованного в категории «Ремонт квартир и домов под ключ».
Твоя задача — определить, упоминает ли селлер какие-либо услуги, которые он готов выполнять ОТДЕЛЬНО, вне комплексного ремонта.

КРИТЕРИИ для ответа True (нужно разделить):
- Селлер явно предлагает отдельные услуги: «также делаем электрику», «сантехнику выполняем отдельно»
- Есть перечисление самостоятельных работ с ценами или условиями для каждой
- Используются фразы: «отдельно», «также выполняем», «помимо основного», «принимаем заказы на»

КРИТЕРИИ для ответа False (не нужно разделять):
- Услуги упомянуты только как составляющие комплексного ремонта: «ремонт под ключ, включая электрику и сантехнику»
- Перечисление видов работ без явного предложения их по отдельности
- Краткое общее описание без конкретики по отдельным услугам

ФОРМАТ ОТВЕТА:
Строго одна строка: True или False
Пример: True
Пример: False

ПРИМЕРЫ:
Опис

  0%|          | 0/20 [00:00<?, ?it/s]

2026-04-07 23:40:49,168 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: начало вызова API
2026-04-07 23:40:49,170 - mistral-call - DEBUG - [MISTRAL 🇫🇷] call_mistral: параметры - ['messages', 'temperature']
2026-04-07 23:40:49,172 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: попытка 1/1 с моделью ministral-14b-latest, ключ 2/21
2026-04-07 23:40:49,196 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: вызов функции с timeout=240
2026-04-07 23:40:49,199 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: попытка 1
2026-04-07 23:40:49,201 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: попытка вызова модели ministral-14b-latest
2026-04-07 23:40:49,634 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: получен ответ длиной 5 символов
2026-04-07 23:40:49,636 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: успешное выполнение на попытке 1
2026-04-07 23:40:49,638 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: успешный вызов модели ministral-14b-latest
2026-04-07 23:40:49,640 - mistral-c

['False',
 'False',
 'False',
 'False',
 'False',
 'False',
 'False',
 'True',
 'True',
 'False',
 'True',
 'False',
 'True',
 'False',
 'True',
 'True',
 'False',
 'False',
 'False',
 'True']

In [22]:
subsample_df["mistral_prediction"] = mistral_predictions
subsample_df[["description", "shouldSplit", "mistral_prediction"]]

,description,shouldSplit,mistral_prediction
1230,"Профессиональный ремонт квартир, коттеджей и о...",False,False
793,все виды отделки,False,False
956,Ремонт квартир и домов под ключ по договору с ...,False,False
261,Бригада из двух человек с опытом работы по отд...,False,False
896,"KОMПЛЕКСНЫЙ pемонт квартиp ""с нуля""!\nСпециали...",False,False
2141,Сделаю всё качественно и аккуратно. Опыт работ...,False,False
269,РемонтСервис64💪\nНА ФОТО НАШИ РЕАЛЬНЫЕ РАБОТЫ!...,False,False
1580,"Ремонт, отделка, кафель, электропроводка, тёпл...",False,True
1601,"ОТКОСЫ ПЛАСТИКОВЫЕ, ЛАМИНИРОВАННЫЕ. Простыe, п...",False,True
1533,Выполним качественный ремонт вашей квартиры и ...,False,False


In [24]:
from sklearn.model_selection import train_test_split

_, stratified_test_df = train_test_split(data_df, test_size=0.2, random_state=42, stratify=data_df["shouldSplit"])
stratified_test_df = stratified_test_df.sample(100, random_state=43)

In [25]:
stratified_test_df["mistral_prediction"] = None
for idx, row in tqdm(stratified_test_df.iterrows(), total=len(stratified_test_df)):
    messages = build_messages(row)
    response = call_mistral(cfg, messages=messages, temperature=0.0)
    stratified_test_df.at[idx, "mistral_prediction"] = response

  0%|          | 0/100 [00:00<?, ?it/s]

2026-04-07 23:42:19,736 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: начало вызова API
2026-04-07 23:42:19,736 - mistral-call - DEBUG - [MISTRAL 🇫🇷] call_mistral: параметры - ['messages', 'temperature']
2026-04-07 23:42:19,736 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: попытка 1/1 с моделью ministral-14b-latest, ключ 2/21
2026-04-07 23:42:19,767 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: вызов функции с timeout=240
2026-04-07 23:42:19,767 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: попытка 1
2026-04-07 23:42:19,771 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: попытка вызова модели ministral-14b-latest
2026-04-07 23:42:20,099 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: получен ответ длиной 5 символов
2026-04-07 23:42:20,100 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: успешное выполнение на попытке 1
2026-04-07 23:42:20,102 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: успешный вызов модели ministral-14b-latest
2026-04-07 23:42:20,105 - mistral-c